# ResNet50-4-Mean Example

In this example, we will evaluate labeling methods on the top 200 neurons of the ResNet50's fourth convolution block outputs.

The 200 neurons are sampled based on the average activation over a probing image dataset (imagenet validation split).

Since the outputs are convolution maps, they are average pooled to get one activation value which will be treated as the target neuron's activation to label then evaluate.

The neurons will be labeld with a curated label set from imagnet21k class names.

The CLIP model used to embed image and text inputs is a CLIP Vit-B/32 (https://huggingface.co/laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K)

The labeling pipelines used in this experiment are:

- SemanticLens: https://arxiv.org/abs/2501.05398
- CLIP-Dissect: https://arxiv.org/abs/2204.10965
- Contrastive Semantic Projection: https://arxiv.org/abs/2604.22477

In [ ]:
# The example requires `open-clip-torch` and `torchvision`.
%pip install open-clip-torch torchvision

## Configs

We set high level configurations for the experiment through the 'load_config' function.

The raw data, precompute data and experiment results are all under the root path of './data/'. 

In [1]:
import torch
from neurolens.config import load_config
from neurolens.utils.path_utils import PathConfigs
import logging

logging.basicConfig(level=logging.ERROR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

CONFIG = load_config(
    {
        "io": {"root_data_dir_path": "../../data"},
        "neuron_data": {
            "topk_neurons": 200,
            "required_sample_count": 100,
            "mean_top_count": 100,
            "ignore_overly_active_neurons": False,
            "allow_neg_reuse": True,
        },
        "evaluation": {"topk_text": 1, "sample_count": 30},
    }
)

PATH_CONFIGS = PathConfigs(CONFIG)

Using device: cuda


## Load Datasets

You can grab the imagenet validation set from https://image-net.org/download.php and you can find the labels dataset under 'datasets/imagenet21k_minimal.txt'. 

The 'coco_control' dataset is a subset of the coco 2014 validation set (https://cocodataset.org/), 500 images are randomly sampled that will later be used in the validation. They represent natural images from a different distrubtion than the probing image dataset.

In [2]:
from neurolens.dataset.image import ImageFolderDataset
from neurolens.dataset.text import TextFileDataset


imagenet_val_img_dataset = ImageFolderDataset(
    identifier="imagenet_val",
    root_dir_path=CONFIG.io.root_data_dir_path / CONFIG.io.raw_data_dir_name / "image/imagenet_val",
)

coco_control_img_dataset = ImageFolderDataset(
    identifier="coco_control",
    root_dir_path=CONFIG.io.root_data_dir_path / CONFIG.io.raw_data_dir_name / "image/coco_test2014_control",
)

imagenet_21k_text_dataset = TextFileDataset(
    identifier="imagenet_21k_minimal",
    file_path=CONFIG.io.root_data_dir_path / CONFIG.io.raw_data_dir_name / "text/imagenet21k_minimal.txt",
)

## Load Image/Text (CLIP) Model

The 'src/clip_vit_b_32_wrapper.py' file includes a simple wrapper of the CLIP Vit-B/32 model. 

The wrapper just exposes the image processor of the model and defines how an input of image features of text is transformed into image embeddings.

In [3]:
from src.clip_vit_b_32_wrapper import CLIPViTB32Wrapper, CLIPViTB32ImgTextModel

clip_vit_b_32_img_text_model = CLIPViTB32ImgTextModel(CLIPViTB32Wrapper(DEVICE))

/home/bouanani/projects/neurolens_test/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Target Model

The target model is a ResNet50 wrapped in 'src/resnet50_mean_target_model.py' to define how activation values from input images are extracted.

In [4]:
from src.resnet50_mean_target_model import ResNet50TargetModel

resnet_target_model = ResNet50TargetModel(4, DEVICE)

## Load Score Functions

We compare four different labeling pipelines:

- SemanticLens: simple mean of highly activating images
- CLIP-Dissect: Using a soft pointwise weighted pointwise mutual information function
- Contrastive Semantic Projection: where we have two variations, one with full projection (gamma=1) and the other one is a softer projection (gamma=0.5)

In [5]:
from neurolens.score_function import (
    SemanticLensScoreFunction,
    ContrastiveProjectionScoreFunction,
    CLIPDissectScoreFunction,
)

semantic_lens = SemanticLensScoreFunction(
    config=CONFIG,
    path_configs=PATH_CONFIGS,
    img_text_model=clip_vit_b_32_img_text_model,
    text_dataset=imagenet_21k_text_dataset,
)

clip_dissect = CLIPDissectScoreFunction(
    config=CONFIG,
    path_configs=PATH_CONFIGS,
    img_text_model=clip_vit_b_32_img_text_model,
    img_dataset=imagenet_val_img_dataset,
    text_dataset=imagenet_21k_text_dataset,
    temp=10,
    lambda_weight=1,
)

contrastive_projection = ContrastiveProjectionScoreFunction(
    config=CONFIG,
    path_configs=PATH_CONFIGS,
    img_text_model=clip_vit_b_32_img_text_model,
    text_dataset=imagenet_21k_text_dataset,
    gamma=1.0,
)

contrastive_projection_gamma = ContrastiveProjectionScoreFunction(
    config=CONFIG,
    path_configs=PATH_CONFIGS,
    img_text_model=clip_vit_b_32_img_text_model,
    text_dataset=imagenet_21k_text_dataset,
    gamma=0.5,
)

SCORE_FUNCTIONS = [semantic_lens, clip_dissect, contrastive_projection, contrastive_projection_gamma]

## Load Neuron Data

Given the experiment configurations (top-200 neurons, etc..), we load the neuron data that includes the representative images from the probing image dataset for each selected neuron alongside their contrastive counterpart (highly similar images that do not activate) and other data (e.g., activation values).

In [6]:
from neurolens.target_model.neuron_data import load_batched_neuron_data

neuron_data = load_batched_neuron_data(
    config=CONFIG,
    path_configs=PATH_CONFIGS,
    target_model=resnet_target_model,
    img_text_model=clip_vit_b_32_img_text_model,
    img_dataset=imagenet_val_img_dataset,
    device=DEVICE,
)

## Evaluate

We finally define and launch two evaluation pipelines:

- CoSy (https://arxiv.org/html/2405.20331v2): Using synthetic images generated by stable diffusion v1.5 (https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), we measure how each assigned label activates the target neuron. The higher the synthetic activation, the better the labeling performance is.

- Simulation Correlation (inspired from https://lilywenglab.github.io/Linear-Explanation/): Given the assigned label, we measure how a simulator (another CLIP model) can simulate the activation of each target neuron on a held-out set from the probing image set and measure its correlation with the true activations on the target model.

In [7]:
from neurolens.evaluation.cosy import CoSyEvaluation, load_cosy_config, CoSyEvaluationColumnNames
from neurolens.evaluation.sim_corr import SimCorrEvaluation, load_sim_corr_config, SimCorrEvaluationColumnNames

cosy_eval = CoSyEvaluation(
    config=CONFIG,
    cosy_config=load_cosy_config({"normalize_activations": True}),
    path_configs=PATH_CONFIGS,
    target_model=resnet_target_model,
    img_text_model=clip_vit_b_32_img_text_model,
    img_dataset=imagenet_val_img_dataset,
    text_dataset=imagenet_21k_text_dataset,
    control_img_dataset=coco_control_img_dataset,
    device=DEVICE,
)

sim_corr_eval = SimCorrEvaluation(
    config=CONFIG,
    sim_corr_config=load_sim_corr_config({"weighted": False, "topk_text": 1}),
    path_configs=PATH_CONFIGS,
    target_model=resnet_target_model,
    img_text_model=clip_vit_b_32_img_text_model,
    img_dataset=imagenet_val_img_dataset,
    text_dataset=imagenet_21k_text_dataset,
    simulator=clip_vit_b_32_img_text_model,
    device=DEVICE,
)


In [8]:
for sf in SCORE_FUNCTIONS:
    text_scores, text_indices = sf.compute_text_scores(neuron_data)

    cosy_eval.evaluate_from_computed_text_scores(
        score_function=sf,
        neuron_data=neuron_data,
        text_scores=text_scores,
        text_indices=text_indices,
    )

    sim_corr_eval.evaluate_from_computed_text_scores(
        score_function=sf,
        neuron_data=neuron_data,
        text_scores=text_scores,
        text_indices=text_indices,
    )

SimCorr evaluation for neuron 609: 100%|██████████| 200/200 [00:00<00:00, 2655.75it/s]


## Aggregate and Show Results

In [9]:
DMA_SCORES = [
    cosy_eval.load_from_csv(score_function=sf)[CoSyEvaluationColumnNames.DMA_SCORE].mean() for sf in SCORE_FUNCTIONS
]
SIM_CORR_SCORES = [
    sim_corr_eval.load_from_csv(score_function=sf)[SimCorrEvaluationColumnNames.SIM_CORR_SCORE].mean()
    for sf in SCORE_FUNCTIONS
]

In [10]:
print("DMA Scores")
for score_function, dma_score in zip(SCORE_FUNCTIONS, DMA_SCORES):
    print(f"{score_function.get_label():<40}: {dma_score:.2f}")

DMA Scores
semantic_lens                           : 0.30
clip_dissect-temp=10-lambda=1           : 0.33
contrastive_projection-gamma=1.0        : 0.35
contrastive_projection-gamma=0.5        : 0.34


In [11]:
print("\nSimCorr Scores")
for score_function, sim_corr_score in zip(SCORE_FUNCTIONS, SIM_CORR_SCORES):
    print(f"{score_function.get_label():<40}: {sim_corr_score:.2f}")


SimCorr Scores
semantic_lens                           : 0.22
clip_dissect-temp=10-lambda=1           : 0.24
contrastive_projection-gamma=1.0        : 0.23
contrastive_projection-gamma=0.5        : 0.23
